<a id="top"></a>

# Demo: tokens are not characters

```yaml
title: "Demo: tokens are not characters"
keywords: tokens, tokenizer, bpe, boundaries, chars per token, teacher demo
estimated duration: 15
```

> **Lesson:** L01. Teacher demo — projected in class. Maps to Demo 1 in the roadmap's [demos_or_activities.md](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md). Runs fully offline.

## Contents

- [1. Setup](#1-setup)
- [2. Token boundaries](#2-token-boundaries)
- [3. Tokenizers disagree](#3-tokenizers-disagree)
- [4. Where the rule of thumb breaks](#4-where-the-rule-of-thumb-breaks)

## 1. Setup

Imports and the four contrast strings. tiktoken downloads its vocab once, then runs offline.

In [1]:
import tiktoken

SAMPLES = {
    "english": "The quick brown fox jumps over the lazy dog.",
    "code": "def f(x): return x ** 2",
    "json": '{"user_id": 12345, "events": ["click", "scroll"]}',
    "non_latin": "こんにちは、世界",  # "hello, world" in Japanese
}


def token_pieces(text: str, encoding_name: str = "cl100k_base") -> list[str]:
    """Decode each token of `text` on its own so we can see the boundaries.

    Example: token_pieces("def f(x)") -> ['def', ' f', '(x', ')']
    """
    enc = tiktoken.get_encoding(encoding_name)
    return [enc.decode([token_id]) for token_id in enc.encode(text)]


def show_boundaries(text: str, encoding_name: str = "cl100k_base") -> str:
    """Render token boundaries joined by '|' for projection."""
    return "|".join(token_pieces(text, encoding_name))


def tiktoken_count(text: str, encoding_name: str = "cl100k_base") -> int:
    return len(tiktoken.get_encoding(encoding_name).encode(text))


print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. Token boundaries

Ask the room (rhetorically) which string is longest in tokens *before* revealing the counts.

In [2]:
for name, text in SAMPLES.items():
    print(f"[{name}] {len(text)} chars -> {tiktoken_count(text)} tokens")
    print("   ", show_boundaries(text))
    print()

[english] 44 chars -> 10 tokens
    The| quick| brown| fox| jumps| over| the| lazy| dog|.

[code] 23 chars -> 9 tokens
    def| f|(x|):| return| x| **| |2

[json] 49 chars -> 17 tokens
    {"|user|_id|":| |123|45|,| "|events|":| ["|click|",| "|scroll|"]}

[non_latin] 8 chars -> 5 tokens
    こんにちは|、|�|�|界



[↑ Back to top](#top)

## 3. Tokenizers disagree

The same string counts differently across tokenizer families. Claude has its own tokenizer too, whose real count shows up in `usage` on a live call.

In [3]:
for name, text in SAMPLES.items():
    cl = tiktoken_count(text, "cl100k_base")
    try:
        o2 = tiktoken_count(text, "o200k_base")
    except Exception:
        # An older tiktoken may not ship the o200k_base vocab; -1 marks "unavailable".
        o2 = -1
    print(f"[{name:9}] cl100k={cl:3}  o200k={o2:3}")

[english  ] cl100k= 10  o200k= 10
[code     ] cl100k=  9  o200k=  9
[json     ] cl100k= 17  o200k= 18
[non_latin] cl100k=  5  o200k=  3


[↑ Back to top](#top)

## 4. Where the rule of thumb breaks

The ≈4-chars-per-token rule holds for English and breaks on JSON.

In [4]:
for name in ("english", "json"):
    text = SAMPLES[name]
    ratio = len(text) / tiktoken_count(text)
    print(f"[{name}] {ratio:.1f} chars/token  (rule of thumb says ~4.0)")

[english] 4.4 chars/token  (rule of thumb says ~4.0)
[json] 2.9 chars/token  (rule of thumb says ~4.0)


[↑ Back to top](#top)